# Sommelier — Vietnamese UV Isolated Venv Test on Kaggle (2x T4 GPU)

Runs the sommelier podcast pipeline inside a **clean isolated `venv`** (`/kaggle/working/sommelier_env`) using **`uv`** on Kaggle with dual T4 GPUs.
This notebook dynamically patches runtime dependencies and handles Kaggle paths without modifying repository source files.


In [3]:
import os
os.environ["MPLBACKEND"] = "Agg"

## 1. Sanity check the Kaggle runtime & GPUs

In [4]:
!nvidia-smi
!python --version
!df -h /kaggle/working 2>/dev/null || df -h .
import torch
print('PyTorch version:', torch.__version__)
print('CUDA version:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('Device count (GPUs):', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}:', torch.cuda.get_device_name(i))


Sat Aug 15 05:35:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Setup Working Directory & Repository

In [5]:
import os
import shutil

BASE_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()
PROJECT_DIR = os.path.join(BASE_DIR, 'sommerlier')
ENV_DIR = os.path.join(BASE_DIR, 'sommelier_env')
AUDIO_DIR = os.path.join(BASE_DIR, 'vi_audio')

print(f"BASE_DIR: {BASE_DIR}")
print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"ENV_DIR: {ENV_DIR}")
print(f"AUDIO_DIR: {AUDIO_DIR}")

# Luôn xóa repo cũ và clone lại
if os.path.exists(PROJECT_DIR):
    print(f"Removing old repository: {PROJECT_DIR}")
    shutil.rmtree(PROJECT_DIR)

os.chdir(BASE_DIR)

print("Cloning repository...")
!git clone https://github.com/foresst123/sommerlier.git

print("Clone completed.")

BASE_DIR: /kaggle/working
PROJECT_DIR: /kaggle/working/sommerlier
ENV_DIR: /kaggle/working/sommelier_env
AUDIO_DIR: /kaggle/working/vi_audio
Cloning repository...
Cloning into 'sommerlier'...
remote: Enumerating objects: 165, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 165 (delta 72), reused 143 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (165/165), 22.35 MiB | 22.50 MiB/s, done.
Resolving deltas: 100% (72/72), done.
Clone completed.


## 3. Install dependencies into isolated venv via UV (~10 min)

Mirrors the three-step install order (torch CUDA 12.6 first, then requirements).


In [6]:
# 1. Install uv package manager
!pip install -q uv yt-dlp

# 2. Create clean isolated virtual environment
!uv venv --allow-existing {ENV_DIR}

# 3. Define proposed optimized dependencies list
PROPOSED_REQUIREMENTS = """
numpy==2.2.2
torch==2.8.0
torchaudio==2.8.0
torchvision==0.23.0
lightning==2.4.0
torchmetrics==1.6.2
onnxruntime-gpu==1.19.2
nemo-toolkit[asr]>=2.2.0
pyannote.audio==4.0.7
speechbrain==1.0.2
faster-whisper==1.2.0
whisperx==3.8.6
ctranslate2==4.5.0
demucs>=4.0.1
panns-inference
librosa==0.10.2.post1
soundfile==0.13.1
pydub==0.25.1
julius==0.2.7
numba==0.61.2
transformers==4.53.0
huggingface-hub>=0.9.8
g2pk
jamo
nltk==3.9.1
openai==1.63.0
tritony==0.0.20
tritonclient[grpc,http]
pandas==2.2.3
PyYAML==6.0.2
tqdm==4.67.1
wandb==0.19.6
requests==2.32.4
einops==0.8.1
hydra-core==1.3.2
omegaconf==2.3.0
yt-dlp
setuptools>=75.0.0
sacrebleu

"""

req_file = os.path.join(BASE_DIR, 'requirements_proposed.txt')
with open(req_file, 'w') as f:
    f.write(PROPOSED_REQUIREMENTS.strip())

print('>>> Step 1: Pre-installing PyTorch CUDA 12.6 inside isolated venv...')
!uv pip install --python {ENV_DIR} torch==2.7.1 torchaudio==2.7.1 torchvision==0.22.1 --extra-index-url https://download.pytorch.org/whl/cu126

print('\n>>> Step 2: Installing proposed dependencies into venv via UV...')
!uv pip install --python {ENV_DIR} -r {req_file} --extra-index-url https://download.pytorch.org/whl/cu126 --index-strategy unsafe-best-match


Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: sommelier_env
Activate with: source sommelier_env/bin/activate
>>> Step 1: Pre-installing PyTorch CUDA 12.6 inside isolated venv...
Using Python 3.12.13 environment at: sommelier_env
Resolved 29 packages in 613ms                                        
Prepared 29 packages in 53.04s                                           
░░░░░░░░░░░░░░░░░░░░ [0/29] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 29 packages in 6.51s                              
 + filelock==3.32.3
 + fsspec==2026.7.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.6.1
 + n

In [7]:
# --- QWEN3-ASR ISOLATED ENVIRONMENT SETUP ---
# We create a separate environment for Qwen3-ASR to avoid huggingface-hub conflicts with WhisperX.
QWEN3_ENV_DIR = os.path.join(BASE_DIR, "qwen3_env")

!uv venv --allow-existing {QWEN3_ENV_DIR} --python 3.12

# Install PyTorch CUDA 12.6
!uv pip install --python {QWEN3_ENV_DIR} torch==2.7.1 torchaudio==2.7.1 torchvision==0.22.1 --extra-index-url https://download.pytorch.org/whl/cu126

# Install Qwen3 specific dependencies (Transformers 5.13 requires newer huggingface-hub)
!uv pip install --python {QWEN3_ENV_DIR} "transformers>=5.13.0" "huggingface-hub>=1.5.0" accelerate soundfile librosa

# Verify installation
!{QWEN3_ENV_DIR}/bin/python -c "from transformers import AutoProcessor, AutoModelForMultimodalLM; print(\"✅ Qwen3 environment OK\")"


Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: qwen3_env
Activate with: source qwen3_env/bin/activate
Using Python 3.12.13 environment at: qwen3_env
Resolved 29 packages in 141ms                                        
░░░░░░░░░░░░░░░░░░░░ [0/29] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 29 packages in 30.82s                             
 + filelock==3.32.3
 + fsspec==2026.7.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.6.1
 + numpy==2.5.2
 + nvidia-cublas-cu12==12.6.4.1
 + nvidia-cuda-cupti-cu12==12.6.80
 + nvidia-cuda-nvrtc-cu12==12.6.77
 + nvidia-cuda-runtime-cu12==12.6.77
 + n

In [8]:
# Write the qwen3_worker.py script to the pipeline directory
worker_code = """#!/usr/bin/env python3
import sys, json, os, torch
import soundfile as sf

def load_model():
    from transformers import AutoProcessor, AutoModelForMultimodalLM
    model_name = "Qwen/Qwen3-ASR-1.7B-hf"
    device = torch.device("cuda:0")  # CUDA_VISIBLE_DEVICES remaps physical GPU 1 to cuda:0
    print(json.dumps({"status": "loading", "model": model_name}), flush=True)
    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModelForMultimodalLM.from_pretrained(model_name, device_map={"": device})
    model.eval()
    print(json.dumps({"status": "ready", "device": str(device)}), flush=True)
    return model, processor, device

def transcribe(model, processor, device, audio_path, language="vi"):
    try:
        audio_data, sr = sf.read(audio_path, dtype="float32")
        if sr != 16000:
            import librosa
            audio_data = librosa.resample(audio_data, orig_sr=sr, target_sr=16000)
        conversation = [{"role": "user", "content": [{"type": "audio", "audio_url": "dummy"}, {"type": "text", "text": "Transcribe the audio in Vietnamese."}]}]
        text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
        inputs = processor(text=text, audio=audio_data, return_tensors="pt", sampling_rate=16000).to(device)
        with torch.no_grad():
            gen_ids = model.generate(**inputs, max_new_tokens=256)
            gen_ids = gen_ids[:, inputs.input_ids.size(1):]
            response = processor.batch_decode(gen_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        return response.strip()
    except Exception as e:
        return f"[ERROR] {e}"

def main():
    model, processor, device = load_model()
    for line in sys.stdin:
        line = line.strip()
        if not line: continue
        try:
            request = json.loads(line)
        except json.JSONDecodeError:
            print(json.dumps({"error": "invalid JSON"}), flush=True)
            continue
        cmd = request.get("cmd", "transcribe")
        if cmd == "quit":
            print(json.dumps({"status": "shutdown"}), flush=True)
            break
        elif cmd == "ping":
            print(json.dumps({"status": "ok"}), flush=True)
            continue
        audio_path = request.get("audio_path", "")
        if not audio_path or not os.path.exists(audio_path):
            print(json.dumps({"error": f"audio file not found: {audio_path}"}), flush=True)
            continue
        text = transcribe(model, processor, device, audio_path, request.get("language", "vi"))
        print(json.dumps({"text": text}), flush=True)

if __name__ == "__main__":
    main()
"""
import glob
# Tự động quét để bắt đúng thư mục dù tên là sommelier hay sommerlier
PROJECT_DIR_DYNAMIC = glob.glob(os.path.join(BASE_DIR, "som*lier"))[0]
worker_path = os.path.join(PROJECT_DIR_DYNAMIC, "podcast-pipeline", "qwen3_worker.py")

with open(worker_path, "w", encoding="utf-8") as f:
    f.write(worker_code)
print(f"✅ Written qwen3_worker.py to {worker_path}")



✅ Written qwen3_worker.py to /kaggle/working/sommerlier/podcast-pipeline/qwen3_worker.py


## 4. Hugging Face authentication & Config update

In [9]:
# Authenticate HuggingFace Token (supports Kaggle Secrets & Interactive Input)
import json, pathlib

hf_token = ""
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print("Fetched HF_TOKEN from Kaggle Secrets.")
except Exception as e:
    print("Kaggle Secrets not available or HF_TOKEN secret missing.")

if not hf_token:
    from getpass import getpass
    hf_token = getpass('Enter HF Token (hf_...): ')

from huggingface_hub import login
login(token=hf_token)

# Update config.json in podcast-pipeline
cfg_path = os.path.join(PROJECT_DIR, 'podcast-pipeline', 'config.json')
if os.path.exists(cfg_path):
    cfg = json.loads(pathlib.Path(cfg_path).read_text())
    cfg['huggingface_token'] = hf_token
    pathlib.Path(cfg_path).write_text(json.dumps(cfg, indent=2, ensure_ascii=False))
    print(f'Updated Hugging Face token in {cfg_path}')


Fetched HF_TOKEN from Kaggle Secrets.
Updated Hugging Face token in /kaggle/working/sommerlier/podcast-pipeline/config.json


## 5. Prepare Audio Input (Auto-detects Kaggle Dataset `/kaggle/input`, Drag-Drop, or Fallback)

In [10]:
import os, glob, random, shutil, pathlib
import torch, torchaudio

# Always prepare a writable working audio folder
BASE_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()
AUDIO_DIR = os.path.join(BASE_DIR, 'vi_audio')
os.makedirs(AUDIO_DIR, exist_ok=True)

# 1. Auto-detect if user attached audio via Kaggle Datasets (/kaggle/input/...)
input_audio_files = []
if os.path.exists('/kaggle/input'):
    extensions = ('*.mp3', '*.wav', '*.flac', '*.m4a', '*.aac', '*.ogg')
    for ext in extensions:
        input_audio_files.extend(glob.glob(f'/kaggle/input/**/{ext}', recursive=True))

if input_audio_files:
    print(f"✅ Detected {len(input_audio_files)} audio file(s) in /kaggle/input/:")
    for f in input_audio_files:
        print("  -", f)
        dst = os.path.join(AUDIO_DIR, os.path.basename(f))
        if not os.path.exists(dst):
            shutil.copy(f, dst)
    print(f"Sync completed into writable folder: {AUDIO_DIR}")

# 2. Scan audio files in /kaggle/working/vi_audio/
audio_extensions = ('.mp3', '.wav', '.flac', '.m4a', '.aac', '.ogg')
audio_files = [f for f in glob.glob(os.path.join(AUDIO_DIR, '*')) if f.lower().endswith(audio_extensions)]

# 3. Fallback: If no audio files exist, generate a 10s synthetic test WAV file
if not audio_files:
    print('\n⚠️ No audio files found in /kaggle/input or /kaggle/working/vi_audio/.')
    print('Generating synthetic benchmark WAV file in vi_audio for smoke testing...')
    sample_rate = 16000
    duration_sec = 10
    t = torch.linspace(0, duration_sec, sample_rate * duration_sec)
    waveform = 0.3 * torch.sin(2 * 3.14159 * 440 * t) + 0.2 * torch.sin(2 * 3.14159 * 880 * t)
    waveform = waveform.unsqueeze(0)
    
    fallback_wav = os.path.join(AUDIO_DIR, 'kaggle_test_sample.wav')
    torchaudio.save(fallback_wav, waveform, sample_rate)
    audio_files = [fallback_wav]
    print(f'✅ Fallback test WAV created at: {fallback_wav}')

print(f'\n✅ Total audio file(s) ready for pipeline: {len(audio_files)}')
for af in audio_files:
    print('  -', af)


✅ Detected 1 audio file(s) in /kaggle/input/:
  - /kaggle/input/datasets/kieuduclamk18hl/data-thu-that-thach/thu_that_thach_10m.mp3
Sync completed into writable folder: /kaggle/working/vi_audio

✅ Total audio file(s) ready for pipeline: 1
  - /kaggle/working/vi_audio/thu_that_thach_10m.mp3


## 6. Dynamic In-Notebook Patches & Run Pipeline on 2x T4 GPUs

In [11]:
# Dynamically apply in-notebook patches without altering original repo files outside this execution
import glob, os

# Patch 1: Dynamically patch pkg_resources in venv site-packages regardless of Python version
site_pkgs = glob.glob(os.path.join(ENV_DIR, 'lib', 'python*', 'site-packages'))
if site_pkgs:
    pkg_res_file = os.path.join(site_pkgs[0], 'pkg_resources.py')
    with open(pkg_res_file, 'w', encoding='utf-8') as f:
        f.write('def declare_namespace(name): pass\n')
    print(f'✅ Dynamic patch applied to {pkg_res_file}')


✅ Dynamic patch applied to /kaggle/working/sommelier_env/lib/python3.12/site-packages/pkg_resources.py
✅ Dynamic patch applied to /kaggle/working/sommerlier/podcast-pipeline/main_original_ASR_MoE.py
✅ Dynamic patch applied to remove chunkformer import in /kaggle/working/sommerlier/podcast-pipeline/main_original_ASR_MoE.py
✅ Patch 7: use_auth_token -> token in /kaggle/working/sommerlier/podcast-pipeline/main_original_ASR_MoE.py
✅ Patch 8: Dual-GPU allocation applied to /kaggle/working/sommerlier/podcast-pipeline/main_original_ASR_MoE.py
✅ Patch 10: Fixed Sortformer Model ID & Inference token applied
✅ Patch 11: Fixed Pyannote 4.x DiarizeOutput applied


In [12]:
import sys
import subprocess

import os
BASE_DIR = '/kaggle/working'
python_bin = os.path.join(BASE_DIR, "sommelier_env", "bin", "python")


code = """
import pyannote.audio
import inspect
from pyannote.audio.pipelines import SpeakerDiarization

print("pyannote.audio version:", pyannote.audio.__version__)
print("pyannote.audio path:", pyannote.audio.__file__)
print("SpeakerDiarization.__init__ signature:")
print(inspect.signature(SpeakerDiarization.__init__))
"""

result = subprocess.run(
    [python_bin, "-c", code],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

pyannote.audio version: 4.0.7
pyannote.audio path: /kaggle/working/sommelier_env/lib/python3.12/site-packages/pyannote/audio/__init__.py
SpeakerDiarization.__init__ signature:
(self, legacy: bool = False, segmentation: Union[pyannote.audio.core.model.Model, str, Mapping] = {'checkpoint': 'pyannote/speaker-diarization-community-1', 'subfolder': 'segmentation'}, segmentation_step: float = 0.1, embedding: Union[pyannote.audio.core.model.Model, str, Mapping] = {'checkpoint': 'pyannote/speaker-diarization-community-1', 'subfolder': 'embedding'}, embedding_exclude_overlap: bool = False, plda: Union[pyannote.audio.core.plda.PLDA, str, Dict] = {'checkpoint': 'pyannote/speaker-diarization-community-1', 'subfolder': 'plda'}, clustering: str = 'VBxClustering', embedding_batch_size: int = 1, segmentation_batch_size: int = 1, der_variant: Optional[dict] = None, token: Optional[str] = None, cache_dir: Union[pathlib.Path, str, NoneType] = None)

STDERR:
/kaggle/working/sommelier_env/lib/python3.12/si

In [13]:
import subprocess
main_script = "/kaggle/working/sommerlier/podcast-pipeline/main_original_ASR_MoE.py"
subprocess.run(["sed", "-i", "s/use_auth_token=/token=/g", main_script])
print("✅ Fixed: use_auth_token -> token")


✅ Fixed: use_auth_token -> token


In [14]:
import subprocess

python_bin = "/kaggle/working/sommelier_env/bin/python"

commands = [
    "import sys; print('Python:', sys.executable)",
    "import torch; print('Torch:', torch.__version__); print('Torch CUDA:', torch.version.cuda); print('GPU:', torch.cuda.device_count())",
    "import ctranslate2; print('CT2:', ctranslate2.__version__); print('CT2 path:', ctranslate2.__file__); print('CT2 CUDA devices:', ctranslate2.get_cuda_device_count())",
    "import faster_whisper; print('faster-whisper:', faster_whisper.__version__)",
]

for code in commands:
    result = subprocess.run(
        [python_bin, "-c", code],
        capture_output=True,
        text=True
    )
    print(">", code)
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)

> import sys; print('Python:', sys.executable)
Python: /kaggle/working/sommelier_env/bin/python

> import torch; print('Torch:', torch.__version__); print('Torch CUDA:', torch.version.cuda); print('GPU:', torch.cuda.device_count())
Torch: 2.8.0+cu126
Torch CUDA: 12.6
GPU: 2

> import ctranslate2; print('CT2:', ctranslate2.__version__); print('CT2 path:', ctranslate2.__file__); print('CT2 CUDA devices:', ctranslate2.get_cuda_device_count())
CT2: 4.5.0
CT2 path: /kaggle/working/sommelier_env/lib/python3.12/site-packages/ctranslate2/__init__.py
CT2 CUDA devices: 2

> import faster_whisper; print('faster-whisper:', faster_whisper.__version__)
faster-whisper: 1.2.0



In [15]:
# Run the pipeline configured for Kaggle 2x T4 GPU
os.chdir(os.path.join(PROJECT_DIR, 'podcast-pipeline'))
python_bin = os.path.join(ENV_DIR, 'bin', 'python')
import sys
site_packages = os.path.join(ENV_DIR, 'lib', f'python{sys.version_info.major}.{sys.version_info.minor}', 'site-packages')
nvidia_lib = f"{site_packages}/nvidia/cudnn/lib:{site_packages}/torch/lib"

import os
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":" + nvidia_lib

!CUDA_VISIBLE_DEVICES=0,1 {python_bin} main_original_ASR_MoE.py \
  --input_folder_path {AUDIO_DIR} \
  --lang vi \
  --vad \
  --dia3 \
  --ASRMoE \
  --no-demucs \
  --whisperx_word_timestamps \
  --no-qwen3omni \
  --no-sepreformer \
  --LLM case_0 \
  --seg_th 0.11 \
  --min_cluster_size 11 \
  --clust_th 0.5 \
  --merge_gap 2


/kaggle/working/sommelier_env/lib/python3.12/site-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/kaggle/working/sommelier_env/lib/python3.12/site-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/kaggle/working/sommelier_env/lib/python3.12/site-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/kaggle/working/sommelier_env/lib/python3.12/site-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
INFO:speechbrain.utils.quirks:Applied quirks (see `speechbrain.utils.quirks`): [allow_tf32, disable_jit_profiling]
INFO:speechbrain.utils.quirks:Excluded quirks specified by the `SB_DISABLE_QUIRKS` environment (comma-separated list): []
INFO:nv_one_logger.exporter.export_con

## 7. Inspect Output

In [ ]:
import glob, json, pathlib

result_folder = os.path.join(AUDIO_DIR, '_final')
json_paths = sorted(glob.glob(f'{result_folder}/**/*.json', recursive=True))
print(f'Found {len(json_paths)} result file(s):')
for p in json_paths:
    print(' -', p)

if json_paths:
    result = json.loads(pathlib.Path(json_paths[0]).read_text())
    print('\nMetadata:')
    print(json.dumps(result.get('metadata', {}), indent=2, ensure_ascii=False))
    print(f"\nFirst 3 segments of {len(result.get('segments', []))}:")
    for seg in result.get('segments', [])[:3]:
        print(json.dumps(seg, indent=2, ensure_ascii=False))


Found 1 result file(s):
 - /kaggle/working/vi_audio/_final/-sepreformer-False-demucs-False-vad-True-diaModel-dia3-initPrompt-True-merge_gap-2.0-seg_th-0.11-cl_min-11-cl-th-0.5-LLM-case_0/thu_that_thach_10m/thu_that_thach_10m.json

Metadata:
{
  "audio_duration_seconds": 599.989,
  "audio_duration_minutes": 9.999816666666668,
  "vad_sortformer": {
    "processing_time_seconds": 23.808685779571533,
    "rt_factor": 0.039681870466911115
  },
  "whisper_large_v3": {
    "processing_time_seconds": 123.4468104839325,
    "rt_factor": 0.20574845619491772
  },
  "total_segments": 175,
  "whisperx_alignment": {
    "processing_time_seconds": 0.0,
    "rt_factor": 0.0,
    "enabled": true
  }
}

First 3 segments of 175:
{
  "start": 0.03096875,
  "end": 5.886593750000001,
  "text": "Hôm nay là em vừa nghe thấy giọng anh Sơn là có một chút không ổn. Có phải là do mình chạy show quá không không?",
  "text_whisper": "Hôm nay là em vừa nghe thấy giọng anh Sơn là có một chút không ổn. Có phải là do m

## Kaggle 2x T4 Troubleshooting & Tips

- **Dual GPU Allocation**: Specified `CUDA_VISIBLE_DEVICES=0,1` for multi-GPU runtime.
- **Kaggle Secrets**: Set `HF_TOKEN` in Kaggle Secrets (Add-ons -> Secrets) so Hugging Face models auto-authenticate.
- **Input Audio**: Supports Kaggle Datasets (`/kaggle/input/...`), direct drag-drop (`/kaggle/working/vi_audio/`), or synthetic benchmark fallback audio.
- **Isolated Venv**: `uv` isolates dependencies in `/kaggle/working/sommelier_env` to avoid Kaggle pre-installed package conflicts.


In [ ]:
import pandas as pd

if json_paths:
    result = json.loads(pathlib.Path(json_paths[0]).read_text())
    audio_name = result.get("metadata", {}).get("audio_name", pathlib.Path(json_paths[0]).stem)
    
    rows = []
    for seg in result.get("segments", []):
        # Format time
        start = seg.get("start", 0)
        end = seg.get("end", 0)
        time_str = f"{start:.2f} - {end:.2f}"
        
        # Get speaker
        speaker = seg.get("speaker", "Unknown")
        
        # Get text (fallback to whisper or ensemble if text is not available)
        text = seg.get("text") or seg.get("text_ensemble") or seg.get("text_whisper") or ""
        
        rows.append({
            "Speaker": speaker,
            "Time": time_str,
            "Audio": audio_name,
            "Text": text
        })
    
    df = pd.DataFrame(rows)
    # Configure pandas display to show full text
    pd.set_option("display.max_colwidth", None)
    display(df)
else:
    print("No results found to display.")


,Speaker,Time,Audio,Text
0,SPEAKER_01,0.03 - 5.89,thu_that_thach_10m,Hôm nay là em vừa nghe thấy giọng anh Sơn là có một chút không ổn. Có phải là do mình chạy show quá không không?
1,SPEAKER_00,6.66 - 8.28,thu_that_thach_10m,[ERROR] Input type (float) and bias type (c10::Half) should be the same
2,SPEAKER_00,8.38 - 14.53,thu_that_thach_10m,Cũng khá là bật. À. Mình vừa chuẩn bị cho cái dự án vừa rồi của mình này xong anh cũng làm việc hơi quá tải.
3,SPEAKER_00,14.75 - 15.74,thu_that_thach_10m,[ERROR] Input type (float) and bias type (c10::Half) should be the same
4,SPEAKER_00,16.05 - 16.52,thu_that_thach_10m,[ERROR] Input type (float) and bias type (c10::Half) should be the same
...,...,...,...,...
170,SPEAKER_00,591.77 - 592.49,thu_that_thach_10m,[ERROR] Input type (float) and bias type (c10::Half) should be the same
171,SPEAKER_00,593.19 - 595.97,thu_that_thach_10m,Có những người sẽ không hiểu vì anh để cái âm lượng nó rất lớn.
172,SPEAKER_00,596.54 - 597.71,thu_that_thach_10m,[ERROR] Input type (float) and bias type (c10::Half) should be the same
173,SPEAKER_00,598.17 - 598.24,thu_that_thach_10m,[ERROR] Input type (float) and bias type (c10::Half) should be the same


In [18]:
!zip -r /kaggle/working/vi_audio_output.zip /kaggle/working/vi_audio


  adding: kaggle/working/vi_audio/ (stored 0%)
  adding: kaggle/working/vi_audio/thu_that_thach_10m.mp3 (deflated 29%)
  adding: kaggle/working/vi_audio/_final/ (stored 0%)
  adding: kaggle/working/vi_audio/_final/-sepreformer-False-demucs-False-vad-True-diaModel-dia3-initPrompt-True-merge_gap-2.0-seg_th-0.11-cl_min-11-cl-th-0.5-LLM-case_0/ (stored 0%)
  adding: kaggle/working/vi_audio/_final/-sepreformer-False-demucs-False-vad-True-diaModel-dia3-initPrompt-True-merge_gap-2.0-seg_th-0.11-cl_min-11-cl-th-0.5-LLM-case_0/thu_that_thach_10m/ (stored 0%)
  adding: kaggle/working/vi_audio/_final/-sepreformer-False-demucs-False-vad-True-diaModel-dia3-initPrompt-True-merge_gap-2.0-seg_th-0.11-cl_min-11-cl-th-0.5-LLM-case_0/thu_that_thach_10m/thu_that_thach_10m.json (deflated 90%)
  adding: kaggle/working/vi_audio/_final/-sepreformer-False-demucs-False-vad-True-diaModel-dia3-initPrompt-True-merge_gap-2.0-seg_th-0.11-cl_min-11-cl-th-0.5-LLM-case_0/thu_that_thach_10m/thu_that_thach_10m/ (stored 0